# AI-Powered Twitter Sentiment Analysis

### Project objective
Build a sentiment analysis tool that classifies tweets as **Positive, Negative, or Neutral** using Natural Language Processing (NLP).

### Technologies
- Python
- NLTK
- Pandas
- Scikit-learn
- TF-IDF
- Logistic Regression
- Matplotlib
- Gradio

### Workflow
**Twitter data → NLTK text preprocessing → train/test split → TF-IDF → Logistic Regression → evaluation → interactive Gradio dashboard**

> This notebook contains the final, focused implementation. Experimental Transformer/RoBERTa cells, duplicate model pipelines, repeated UI versions, backup checks, and temporary debugging cells have been intentionally left out.

## 1. Install and import required libraries

Only libraries needed for the final project are included here. The final model is the verified **TF-IDF + Logistic Regression** pipeline.

In [ ]:
!pip -q install nltk

import re
import json
import pickle
import requests
import nltk
import pandas as pd
import matplotlib.pyplot as plt

from io import StringIO
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

nltk.download("stopwords")

print("Libraries loaded successfully.")


## 2. Load the Twitter dataset

The notebook uses the same Twitter dataset and source used in the original project.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/tmun5718/Twitter-Sentiment-Analysis/master/DataSet/FinalizedFull.csv"

response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()

data = pd.read_csv(StringIO(response.text))

print("Dataset loaded successfully.")
print("Shape:", data.shape)
print("Columns:", data.columns.tolist())

display(data.head())

## 3. Understand the dataset

The original labels are encoded as:
- `0` → Negative
- `2` → Neutral
- `4` → Positive

We convert them to readable class names for the rest of the project.

In [ ]:
print("Missing values:")
print(data.isnull().sum())

print("\nDuplicate tweets:", data["tweet"].duplicated().sum())
print("Duplicate rows:", data.duplicated().sum())

label_mapping = {
    0: "negative",
    2: "neutral",
    4: "positive"
}

data["sentiment"] = data["senti"].map(label_mapping)

print("\nSentiment distribution:")
print(data["sentiment"].value_counts())

## 4. Sentiment distribution

A quick visualization helps us understand how the three classes are represented in the dataset.

In [ ]:
sentiment_counts = data["sentiment"].value_counts().reindex(
    ["negative", "neutral", "positive"]
)

plt.figure(figsize=(7, 5))
plt.bar(
    sentiment_counts.index.str.capitalize(),
    sentiment_counts.values
)
plt.title("Twitter Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Tweets")
plt.tight_layout()
plt.show()

## 5. NLTK-based Twitter text preprocessing

NLTK stopwords are used as part of the NLP preprocessing stage.

The same cleaning approach used to train the verified Logistic Regression model is retained so that the saved model and deployment pipeline remain consistent.

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_tweet(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove Twitter mentions
    text = re.sub(r"@\w+", "", text)

    # Remove retweet marker
    text = re.sub(r"\brt\b", "", text)

    # Keep hashtag words while removing the # symbol
    text = re.sub(r"#", "", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Remove English stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

print("NLTK preprocessing function created.")

## 6. Verify preprocessing

Before applying preprocessing to the full dataset, inspect a few real examples.

In [ ]:
for i in range(min(5, len(data))):
    original = data.loc[i, "tweet"]
    cleaned = clean_tweet(original)

    print(f"Tweet {i + 1}")
    print("-" * 70)
    print("Original:", original)
    print("Cleaned :", cleaned)
    print()

In [ ]:
data["cleaned_tweet"] = data["tweet"].apply(clean_tweet)

print("All tweets cleaned successfully.")
display(data[["tweet", "cleaned_tweet", "sentiment"]].head())

## 7. Train/test split

An 80/20 stratified split is used so that all three sentiment classes remain represented in both sets.

In [ ]:
X = data["cleaned_tweet"]
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

## 8. Convert tweets into TF-IDF features

TF-IDF converts cleaned text into numerical features. The vectorizer is fitted **only on the training data** and then used to transform the test data.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF conversion completed.")
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix :", X_test_tfidf.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

## 9. Train the sentiment classifier

Logistic Regression is used as the final classifier because it supports multi-class classification and probability estimates for the interactive tool.

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Logistic Regression trained successfully.")
print("Classes:", list(model.classes_))

## 10. Evaluate the model

The final model is evaluated using accuracy, weighted precision, weighted recall, and weighted F1-score.

In [ ]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    y_test, y_pred,
    average="weighted",
    zero_division=0
)

print("=" * 60)
print("FINAL MODEL PERFORMANCE")
print("=" * 60)
print(f"Accuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-Score : {f1:.4f} ({f1 * 100:.2f}%)")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=["negative", "neutral", "positive"],
        target_names=["Negative", "Neutral", "Positive"],
        zero_division=0
    )
)

## 11. Confusion matrix

The confusion matrix shows where the classifier correctly predicts a class and where it confuses one sentiment with another.

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["negative", "neutral", "positive"]
)

plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Twitter Sentiment Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.xticks(range(3), ["Negative", "Neutral", "Positive"])
plt.yticks(range(3), ["Negative", "Neutral", "Positive"])

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(col, row, cm[row, col], ha="center", va="center")

plt.colorbar(label="Number of Tweets")
plt.tight_layout()
plt.show()

print("Confusion matrix:")
print(cm)

## 12. Test the trained model on new tweets

These examples demonstrate the actual prediction workflow before deployment.

In [ ]:
test_tweets = [
    "I absolutely love this! It is amazing and wonderful!",
    "This is terrible. I hate it and it was a complete disaster.",
    "The product arrived today. Nothing special, just an ordinary experience.",
    "Amazing service! Very happy with the experience.",
    "Worst experience ever. Completely disappointing."
]

print("=" * 60)
print("NEW TWEET PREDICTIONS")
print("=" * 60)

for tweet in test_tweets:
    cleaned = clean_tweet(tweet)
    vector = tfidf.transform([cleaned])

    prediction = model.predict(vector)[0]
    probabilities = model.predict_proba(vector)[0]
    confidence = max(probabilities) * 100

    print("\nTweet:", tweet)
    print("Sentiment:", prediction.capitalize())
    print(f"Confidence: {confidence:.2f}%")
    print("Probabilities:")

    for class_name, probability in zip(model.classes_, probabilities):
        print(f"  {class_name.capitalize():8s}: {probability * 100:.2f}%")

## 13. Save the trained model and TF-IDF vectorizer

These two files are the only model artifacts required by the final dashboard.

In [ ]:
MODEL_FILE = "twitter_sentiment_model.pkl"
VECTORIZER_FILE = "twitter_tfidf_vectorizer.pkl"

with open(MODEL_FILE, "wb") as f:
    pickle.dump(model, f)

with open(VECTORIZER_FILE, "wb") as f:
    pickle.dump(tfidf, f)

print("Saved successfully:")
print("-", MODEL_FILE)
print("-", VECTORIZER_FILE)

## 14. Final model reload check

The deployment should use the saved artifacts rather than depending on variables from the training session. This cell verifies that both files can be loaded successfully.

In [ ]:
with open(MODEL_FILE, "rb") as f:
    saved_model = pickle.load(f)

with open(VECTORIZER_FILE, "rb") as f:
    saved_tfidf = pickle.load(f)

test_text = "I really enjoyed this experience!"
test_vector = saved_tfidf.transform([clean_tweet(test_text)])
test_prediction = saved_model.predict(test_vector)[0]

print("Saved model loaded:", type(saved_model).__name__)
print("Saved vectorizer loaded:", type(saved_tfidf).__name__)
print("Reload test:", test_prediction.capitalize())
print("Deployment artifacts verified.")

# 15. GitHub Pages Deployment Dashboard

The final application is exported as a **single self-contained `index.html` file**.

Unlike the temporary Gradio link, this version does not require a Python server, Render, Hugging Face Spaces, or a running Colab session. The trained TF-IDF + Logistic Regression model is converted into browser-readable data, and prediction is performed locally with JavaScript.

This is suitable for GitHub Pages, which hosts static HTML/CSS/JavaScript files.

In [ ]:

# ============================================================
# EXPORT A SELF-CONTAINED GITHUB PAGES APPLICATION
# ============================================================

from pathlib import Path
import json

GITHUB_APP_DIR = Path("twitter_sentiment_github_pages")
GITHUB_APP_DIR.mkdir(exist_ok=True)

INDEX_FILE = GITHUB_APP_DIR / "index.html"

# Exact trained artifacts
model_for_web = model
tfidf_for_web = tfidf

model_classes = [str(x) for x in model_for_web.classes_]
vocabulary = {str(k): int(v) for k, v in tfidf_for_web.vocabulary_.items()}
idf_values = [float(x) for x in tfidf_for_web.idf_]
coefficients = [[float(x) for x in row] for row in model_for_web.coef_]
intercepts = [float(x) for x in model_for_web.intercept_]
web_stop_words = sorted(str(x) for x in stop_words)

web_model = {
    "classes": model_classes,
    "vocabulary": vocabulary,
    "idf": idf_values,
    "coef": coefficients,
    "intercept": intercepts,
    "stop_words": web_stop_words,
    "ngram_range": [1, 2],
    "feature_count": len(vocabulary),
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }
}

MODEL_JSON = json.dumps(web_model, ensure_ascii=False, separators=(",", ":"))

HTML = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Twitter Sentiment Analysis</title>
<style>
:root{--bg:#070b16;--panel:#0e1526;--border:#26344f;--text:#f8fafc;--muted:#94a3b8;--violet:#8b5cf6;--green:#22c55e;--red:#ef4444;--yellow:#eab308}
*{box-sizing:border-box}
body{margin:0;font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;background:radial-gradient(circle at 15% 10%,rgba(124,58,237,.13),transparent 28%),radial-gradient(circle at 85% 20%,rgba(37,99,235,.12),transparent 30%),var(--bg);color:var(--text);min-height:100vh}
.container{width:min(1120px,calc(100% - 36px));margin:0 auto;padding:34px 0 48px}
.hero{position:relative;overflow:hidden;text-align:center;padding:58px 28px;border:1px solid #334155;border-radius:26px;background:linear-gradient(135deg,#111827,#1e1b4b 55%,#312e81);box-shadow:0 24px 70px rgba(0,0,0,.25)}
.hero::before{content:"";position:absolute;width:250px;height:250px;border-radius:50%;background:rgba(139,92,246,.12);right:-90px;top:-130px}
.hero h1{position:relative;margin:0;font-size:clamp(32px,5vw,52px);line-height:1.08;letter-spacing:-1.8px}
.hero p{position:relative;max-width:760px;margin:18px auto 0;color:#cbd5e1;font-size:16px;line-height:1.7}
.section-title{margin:34px 0 17px;font-size:25px;font-weight:800}
.workspace{display:grid;grid-template-columns:1fr 1fr;gap:18px}
.card{background:rgba(14,21,38,.88);border:1px solid var(--border);border-radius:21px;padding:23px;box-shadow:0 14px 35px rgba(0,0,0,.16)}
.card h2{margin:0 0 17px;font-size:18px}
textarea{width:100%;min-height:215px;resize:vertical;border:1px solid #334155;border-radius:14px;padding:17px;background:#0a1120;color:var(--text);font:inherit;line-height:1.6;outline:none}
textarea:focus{border-color:var(--violet);box-shadow:0 0 0 3px rgba(139,92,246,.13)}
textarea::placeholder{color:#64748b}
.actions{display:flex;gap:10px;margin-top:13px}
button{flex:1;border:0;border-radius:11px;padding:13px 16px;font:inherit;font-weight:750;cursor:pointer}
.analyze{color:white;background:linear-gradient(135deg,#7c3aed,#4f46e5)}
.clear{color:#cbd5e1;background:#1e293b}
.result{min-height:215px;display:flex;flex-direction:column;justify-content:center}
.ready{text-align:center;color:var(--muted)}
.ready-icon{width:52px;height:52px;margin:0 auto 13px;display:grid;place-items:center;border-radius:15px;color:#c4b5fd;background:rgba(139,92,246,.12);font-size:23px}
.result-label{color:#64748b;font-size:11px;font-weight:800;letter-spacing:1px}
.result-main{display:flex;align-items:center;gap:15px;margin-top:13px}
.result-badge{width:55px;height:55px;border-radius:16px;display:grid;place-items:center;font-size:23px;font-weight:900}
.result-sentiment{font-size:29px;font-weight:900}
.result-status{color:var(--muted);font-size:13px;margin-top:3px}
.positive .result-badge{color:#4ade80;background:rgba(34,197,94,.12)}
.positive .result-sentiment{color:#4ade80}
.negative .result-badge{color:#f87171;background:rgba(239,68,68,.12)}
.negative .result-sentiment{color:#f87171}
.neutral .result-badge{color:#facc15;background:rgba(234,179,8,.12)}
.neutral .result-sentiment{color:#facc15}
.confidence{margin-top:22px}
.confidence-head,.prob-head{display:flex;justify-content:space-between;color:var(--muted);font-size:13px}
.track{height:7px;margin-top:8px;overflow:hidden;border-radius:20px;background:#1e293b}
.fill{height:100%;border-radius:20px}
.positive-fill{background:var(--green)}.negative-fill{background:var(--red)}.neutral-fill{background:var(--yellow)}
.probabilities{margin-top:15px;padding:17px;border-radius:15px;background:#0a1120;border:1px solid #1f2d45}
.prob-title{font-size:14px;font-weight:800;margin-bottom:15px}
.prob-row{margin-top:13px}
.prob-fill{height:6px;margin-top:7px;border-radius:20px}
.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:14px}
.metric{text-align:center;padding:20px 10px;border:1px solid var(--border);border-radius:17px;background:rgba(14,21,38,.88)}
.metric-value{color:#a78bfa;font-size:29px;font-weight:900}
.metric-label{color:var(--muted);font-size:12px;margin-top:5px}
.pipeline{padding:21px;text-align:center;border:1px solid var(--border);border-radius:18px;background:linear-gradient(135deg,rgba(124,58,237,.08),rgba(59,130,246,.06));color:#c4b5fd;font-weight:750}
.footer{margin-top:30px;padding-top:20px;border-top:1px solid #1e293b;text-align:center;color:#64748b;font-size:12px}
@media(max-width:780px){.workspace{grid-template-columns:1fr}.metrics{grid-template-columns:repeat(2,1fr)}.hero{padding:43px 20px}}
@media(max-width:460px){.container{width:min(100% - 22px,1120px)}.metrics{grid-template-columns:1fr 1fr;gap:9px}}
</style>
</head>
<body>
<div class="container">
<section class="hero">
<h1>Twitter Sentiment Analysis</h1>
<p>A Natural Language Processing and Machine Learning tool for classifying tweets as Positive, Neutral, or Negative.</p>
</section>

<div class="section-title">Sentiment Analysis</div>
<section class="workspace">
<div class="card">
<h2>Enter a Tweet</h2>
<textarea id="tweetInput" maxlength="1000" placeholder="Type or paste a tweet here..."></textarea>
<div class="actions">
<button class="analyze" onclick="analyzeTweet()">Analyze Sentiment</button>
<button class="clear" onclick="clearDashboard()">Clear</button>
</div>
</div>

<div class="card">
<h2>Analysis Result</h2>
<div id="result" class="result">
<div class="ready"><div class="ready-icon">✦</div><strong>Ready to analyze</strong><div style="margin-top:7px;">Enter a tweet to see the predicted sentiment.</div></div>
</div>
</div>
</section>

<div class="section-title">Model Performance</div>
<section class="metrics">
<div class="metric"><div class="metric-value" id="accuracyMetric"></div><div class="metric-label">Accuracy</div></div>
<div class="metric"><div class="metric-value" id="precisionMetric"></div><div class="metric-label">Weighted Precision</div></div>
<div class="metric"><div class="metric-value" id="recallMetric"></div><div class="metric-label">Weighted Recall</div></div>
<div class="metric"><div class="metric-value" id="f1Metric"></div><div class="metric-label">Weighted F1 Score</div></div>
</section>

<div class="section-title">NLP & Machine Learning Pipeline</div>
<div class="pipeline">NLTK &nbsp;→&nbsp; Text Preprocessing &nbsp;→&nbsp; TF-IDF &nbsp;→&nbsp; Logistic Regression &nbsp;→&nbsp; Sentiment</div>

<div class="footer">Twitter Sentiment Analysis · Python · NLTK · Scikit-learn · GitHub Pages</div>
</div>

<script>
const MODEL = __MODEL_DATA__;

const LABELS = {negative:"Negative",neutral:"Neutral",positive:"Positive"};

function cleanTweet(text){
    text=String(text).toLowerCase();
    text=text.replace(/http\S+|www\S+|https\S+/g,"");
    text=text.replace(/@\w+/g,"");
    text=text.replace(/\brt\b/g,"");
    text=text.replace(/#/g,"");
    text=text.replace(/[^a-zA-Z\s]/g," ");
    text=text.replace(/\s+/g," ").trim();
    return text.split(/\s+/).filter(Boolean).filter(word=>!MODEL.stop_words.includes(word)).join(" ");
}

function createFeatures(cleanedText){
    const tokens=cleanedText.split(/\s+/).filter(token=>token.length>=2);
    const terms=[];
    for(let i=0;i<tokens.length;i++){
        terms.push(tokens[i]);
        if(i<tokens.length-1) terms.push(tokens[i]+" "+tokens[i+1]);
    }
    const counts={};
    for(const term of terms){
        if(Object.prototype.hasOwnProperty.call(MODEL.vocabulary,term)){
            counts[term]=(counts[term]||0)+1;
        }
    }
    const vector=new Float64Array(MODEL.idf.length);
    for(const [term,count] of Object.entries(counts)){
        const index=MODEL.vocabulary[term];
        vector[index]=count*MODEL.idf[index];
    }
    let norm=0;
    for(let i=0;i<vector.length;i++) norm+=vector[i]*vector[i];
    norm=Math.sqrt(norm);
    if(norm>0) for(let i=0;i<vector.length;i++) vector[i]/=norm;
    return vector;
}

function predict(vector){
    const scores=new Array(MODEL.classes.length).fill(0);
    for(let c=0;c<MODEL.classes.length;c++){
        let score=MODEL.intercept[c];
        const weights=MODEL.coef[c];
        for(let i=0;i<vector.length;i++){
            if(vector[i]!==0) score+=vector[i]*weights[i];
        }
        scores[c]=score;
    }
    const maxScore=Math.max(...scores);
    const exps=scores.map(score=>Math.exp(score-maxScore));
    const total=exps.reduce((sum,value)=>sum+value,0);
    const probabilities=exps.map(value=>value/total);
    let bestIndex=0;
    for(let i=1;i<probabilities.length;i++){
        if(probabilities[i]>probabilities[bestIndex]) bestIndex=i;
    }
    return {className:MODEL.classes[bestIndex],probabilities};
}

function probabilityRow(label,value,fillClass){
    return `<div class="prob-row"><div class="prob-head"><span>${label}</span><strong>${value.toFixed(2)}%</strong></div><div class="track"><div class="fill ${fillClass}" style="width:${value}%"></div></div></div>`;
}

function analyzeTweet(){
    const input=document.getElementById("tweetInput");
    const result=document.getElementById("result");
    const original=input.value.trim();

    if(!original){
        result.innerHTML=`<div class="ready"><div class="ready-icon">✦</div><strong>Ready to analyze</strong><div style="margin-top:7px;">Enter a tweet to see the predicted sentiment.</div></div>`;
        return;
    }

    const cleaned=cleanTweet(original);

    if(!cleaned){
        result.innerHTML=`<div class="ready"><strong>No usable text</strong><div style="margin-top:7px;">Please enter a tweet containing meaningful text.</div></div>`;
        return;
    }

    const vector=createFeatures(cleaned);
    const prediction=predict(vector);
    const probabilities={};

    MODEL.classes.forEach((className,index)=>{
        probabilities[className]=prediction.probabilities[index]*100;
    });

    const predicted=prediction.className;
    const displayName=LABELS[predicted];
    const confidence=probabilities[predicted];

    let accent="neutral";
    let symbol="N";
    if(predicted==="positive"){accent="positive";symbol="P";}
    else if(predicted==="negative"){accent="negative";symbol="N";}

    result.innerHTML=`
    <div class="${accent}">
    <div class="result-label">PREDICTED SENTIMENT</div>
    <div class="result-main">
    <div class="result-badge">${symbol}</div>
    <div><div class="result-sentiment">${displayName}</div><div class="result-status">Sentiment classification completed</div></div>
    </div>
    <div class="confidence">
    <div class="confidence-head"><span>Model confidence</span><strong>${confidence.toFixed(2)}%</strong></div>
    <div class="track"><div class="fill ${accent}-fill" style="width:${confidence}%"></div></div>
    </div>
    <div class="probabilities">
    <div class="prob-title">Sentiment Distribution</div>
    ${probabilityRow("Negative",probabilities.negative||0,"negative-fill")}
    ${probabilityRow("Neutral",probabilities.neutral||0,"neutral-fill")}
    ${probabilityRow("Positive",probabilities.positive||0,"positive-fill")}
    </div>
    </div>`;
}

function clearDashboard(){
    document.getElementById("tweetInput").value="";
    document.getElementById("result").innerHTML=`<div class="ready"><div class="ready-icon">✦</div><strong>Ready to analyze</strong><div style="margin-top:7px;">Enter a tweet to see the predicted sentiment.</div></div>`;
}

document.getElementById("accuracyMetric").textContent=(MODEL.metrics.accuracy*100).toFixed(2)+"%";
document.getElementById("precisionMetric").textContent=(MODEL.metrics.precision*100).toFixed(2)+"%";
document.getElementById("recallMetric").textContent=(MODEL.metrics.recall*100).toFixed(2)+"%";
document.getElementById("f1Metric").textContent=(MODEL.metrics.f1*100).toFixed(2)+"%";
</script>
</body>
</html>
'''

HTML = HTML.replace("__MODEL_DATA__", MODEL_JSON)
INDEX_FILE.write_text(HTML, encoding="utf-8")

README = f'''
# Twitter Sentiment Analysis

A browser-based sentiment analysis tool built from the verified NLTK + TF-IDF + Logistic Regression pipeline.

## Problem statement

Build a sentiment analysis tool to classify tweets as **Positive, Negative, or Neutral** using Natural Language Processing libraries such as NLTK or spaCy.

## Implementation

- Twitter sentiment dataset
- NLTK text preprocessing
- TF-IDF feature extraction
- Logistic Regression classifier
- Negative / Neutral / Positive classes
- Interactive browser dashboard
- GitHub Pages deployment
- No Python server required for the deployed application

## Model performance

- Accuracy: {accuracy * 100:.2f}%
- Weighted Precision: {precision * 100:.2f}%
- Weighted Recall: {recall * 100:.2f}%
- Weighted F1 Score: {f1 * 100:.2f}%

## Deployment

Publish `index.html` with GitHub Pages.

The model parameters are embedded in the page so the browser can perform prediction without a Python backend.
'''

(GITHUB_APP_DIR / "README.md").write_text(README, encoding="utf-8")
(GITHUB_APP_DIR / ".nojekyll").write_text("", encoding="utf-8")

print("=" * 64)
print("GITHUB PAGES APPLICATION CREATED")
print("=" * 64)
print()
print("Folder:", GITHUB_APP_DIR.resolve())
print("Files:")
for file in sorted(GITHUB_APP_DIR.iterdir()):
    print(f"  - {file.name:12s} {file.stat().st_size / 1024:.1f} KB")
print()
print("Main deployment file:", INDEX_FILE.resolve())
print("The final application is self-contained.")
print("=" * 64)


## 16. Verify the GitHub Pages files

Run the next cell after exporting the application. It checks that the deployment file exists and contains the model data.

In [ ]:

from pathlib import Path

index_path = Path("twitter_sentiment_github_pages/index.html")

print("=" * 64)
print("GITHUB PAGES DEPLOYMENT CHECK")
print("=" * 64)

if index_path.exists():
    size_kb = index_path.stat().st_size / 1024
    html = index_path.read_text(encoding="utf-8")

    checks = {
        "index.html exists": True,
        "model data embedded": "const MODEL =" in html,
        "three sentiment classes present": all(
            x in html for x in ["negative", "neutral", "positive"]
        ),
        "static HTML present": "<!DOCTYPE html>" in html,
        "no Gradio dependency": "gradio" not in html.lower()
    }

    for name, passed in checks.items():
        print(("✅ " if passed else "❌ ") + name)

    print(f"\nFile size: {size_kb:.1f} KB")
else:
    print("❌ index.html was not created.")

print("=" * 64)


## 17. Publish the application with GitHub Pages

After running all notebook cells, the folder `twitter_sentiment_github_pages` will contain the deployable website.

Upload the **contents** of that folder to a GitHub repository so that `index.html` is at the repository root (or in the selected Pages source folder).

Then enable **GitHub Pages → Deploy from a branch → main → /(root)**.

The resulting `github.io` address is a persistent GitHub Pages site rather than a temporary Colab/Gradio tunnel.

The deployed application performs inference in the browser because GitHub Pages serves static HTML/CSS/JavaScript and does not run Python or scikit-learn server-side.

### Final requirement check

| Problem statement requirement | Final implementation |
|---|---|
| Sentiment analysis tool | GitHub Pages interactive dashboard |
| Positive classification | Logistic Regression |
| Negative classification | Logistic Regression |
| Neutral classification | Logistic Regression |
| NLP library | NLTK |
| Twitter data | Twitter sentiment dataset |
| Permanent web deployment | GitHub Pages static site |
